# Robo Fundamento — alocacao entre classes com rebalanceamento periodico

**Desafio Itau Asset Quant AI 2026**

> Todo mes o robo decide quanto colocar em cada tipo de investimento — renda fixa, acoes Brasil, bolsa global, ouro e dolar — dando mais peso ao que esta indo bem, e fugindo 100% para a renda fixa quando nada esta.

Nao e escolher acoes: e **alocacao entre classes de ativo**, que e o que uma gestora faz. O rebalanceamento periodico e o mecanismo central.

**Regua = CDI.** No Brasil dinheiro parado rende ~10% ao ano sem risco. Qualquer metrica que ignore o CDI esta inflada.

> Rode tudo: Ambiente de execucao -> Executar tudo (Ctrl+F9). Auto-contido.

In [ ]:
!pip -q install yfinance scipy
import numpy as np, pandas as pd, requests, yfinance as yf
import matplotlib.pyplot as plt
from scipy import stats
plt.rcParams['figure.figsize'] = (11, 4.5); plt.rcParams['axes.grid'] = True; plt.rcParams['grid.alpha'] = 0.25

## 1. Dados — 5 classes de ativo

| Classe | O que e | Papel |
|---|---|---|
| CDI | renda fixa pos-fixada (Banco Central) | o caixa, o porto seguro |
| Acoes | Ibovespa | crescimento Brasil |
| Global | S&P 500 em reais | crescimento internacional (IVVB11) |
| Ouro | ouro em reais | protecao contra crise |
| Dolar | USD/BRL | protecao contra risco Brasil |

In [ ]:
def sgs(codigo):
    """Serie do Banco Central (API SGS). Limite ~10 anos/requisicao -> busca em blocos."""
    partes = []
    for ini, fim in [('01/01/2009','31/12/2016'), ('01/01/2017','31/12/2026')]:
        u = f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados?formato=json&dataInicial={ini}&dataFinal={fim}'
        partes += requests.get(u, timeout=60, headers={'User-Agent':'Mozilla/5.0'}).json()
    d = pd.DataFrame(partes)
    d['data'] = pd.to_datetime(d['data'], format='%d/%m/%Y')
    d['valor'] = d['valor'].astype(float)/100
    return d.set_index('data')['valor'].sort_index()

cdi_m = (1 + sgs(12)).resample('ME').prod() - 1          # CDI diario -> mensal
# ^SP500TR = S&P 500 TOTAL RETURN (inclui dividendos). O indice de preco (^GSPC)
# subestima a classe em ~2% ao ano — quem compra IVVB11 recebe os dividendos.
px = yf.download(['^BVSP','^SP500TR','GC=F','BRL=X'], start='2010-01-01',
                 auto_adjust=True, progress=False)['Close']
pm = px.resample('ME').last(); usd = pm['BRL=X']

niveis = pd.DataFrame({'Acoes': pm['^BVSP'], 'Global': pm['^SP500TR']*usd,
                       'Ouro': pm['GC=F']*usd, 'Dolar': usd})
niveis['CDI'] = (1 + cdi_m.reindex(pm.index).fillna(0)).cumprod()
CLASSES = ['CDI','Acoes','Global','Ouro','Dolar']
RISCO = ['Acoes','Global','Ouro','Dolar']
rets = niveis[RISCO].pct_change(); rets['CDI'] = cdi_m.reindex(pm.index)
rets = rets[CLASSES].dropna()
if rets.index[-1].month == pd.Timestamp.today().month: rets = rets.iloc[:-1]  # mes incompleto
print(f'{len(rets)} meses, de {rets.index.min().date()} a {rets.index.max().date()}')

print('\nRetorno anualizado de cada classe:')
for c in CLASSES:
    r = rets[c]; print(f'  {c:7s} {(1+r).prod()**(12/len(r))-1:>6.1%} a.a.   vol {r.std()*np.sqrt(12):>5.1%}')

Repare: o **Ibovespa perdeu para o CDI**. E por isso que o cardapio de classes importa tanto — sem bolsa global e ouro, o robo ficaria preso a opcoes ruins.

## 2. O sinal e a regra de alocacao

**Momentum de 12 meses** de cada classe, sempre com `shift(1)` (a decisao do mes t so enxerga ate t-1 — sem olhar o futuro).

**Duas travas:**
1. *Momentum absoluto* — classe de risco so entra se render mais que o CDI. Se nenhuma render, o robo fica **100% em CDI**.
2. *Momentum relativo* — entre as aprovadas, a melhor pesa mais: **40 / 30 / 20 / 10%**. O que sobra vai para o CDI.

In [ ]:
ESCALA = [0.40, 0.30, 0.20, 0.10]
mom = (niveis[CLASSES]/niveis[CLASSES].shift(12) - 1).shift(1).reindex(rets.index)

def pesos_ranking():
    linhas = []
    for dt in rets.index:
        m = mom.loc[dt]; p = {c: 0.0 for c in CLASSES}
        if m.isna().any():
            p['CDI'] = 1.0; linhas.append(p); continue
        aprov = sorted([c for c in RISCO if m[c] > m['CDI']], key=lambda c: -m[c])
        for peso, c in zip(ESCALA, aprov): p[c] = peso
        p['CDI'] = 1 - sum(p[c] for c in RISCO)
        linhas.append(p)
    return pd.DataFrame(linhas, index=rets.index)[CLASSES]

def pesos_iguais():
    return pd.DataFrame([{c: 1/len(CLASSES) for c in CLASSES}]*len(rets), index=rets.index)[CLASSES]

pesos_ranking().tail(3).round(2)

## 3. O motor de backtest (implementado por nos)

Mes a mes: aplica os pesos, paga o **custo do giro**, e deixa a carteira **derivar** ate o proximo rebalanceamento.

In [ ]:
CUSTO = 0.0010  # 10 bps por unidade de giro

def backtest(alvo, freq=1, custo=CUSTO):
    efetivos = alvo.iloc[0].copy(); out = []
    for i, dt in enumerate(rets.index):
        giro = float((alvo.loc[dt] - efetivos).abs().sum()) if i % freq == 0 else 0.0
        if i % freq == 0: efetivos = alvo.loc[dt].copy()
        r = rets.loc[dt]
        out.append(float((efetivos*r).sum()) - giro*custo)
        efetivos = efetivos*(1+r); efetivos /= efetivos.sum()   # deriva
    return pd.Series(out, index=rets.index)

def avaliar(r, cdi):
    d = pd.concat([r.rename('r'), cdi.rename('cdi')], axis=1).dropna(); exc = d['r']-d['cdi']; n = len(d)
    eq = (1+d['r']).cumprod(); t, p = stats.ttest_1samp(exc, 0)
    return {'retorno': (1+d['r']).prod()**(12/n)-1, 'vol': d['r'].std()*np.sqrt(12),
            'exCDI': (1+exc).prod()**(12/n)-1, 'sharpe': exc.mean()/exc.std()*np.sqrt(12),
            'maxDD': (eq/eq.cummax()-1).min(), 't': t, 'p': p}

## 4. Resultado

Backtest a partir de **2015** (quando o IVVB11 tornou a bolsa global investivel na B3).

In [ ]:
INICIO = '2015-01-01'
jan = rets.index >= INICIO
cdi = rets['CDI']
carteiras = {'Robo Fundamento (ranking)': backtest(pesos_ranking()),
             'Rebalanceamento puro':      backtest(pesos_iguais()),
             'Ibovespa':                  rets['Acoes'],
             'CDI (regua)':               cdi}

print(f"{'estrategia':28s}{'retorno':>9s}{'vol':>7s}{'exCDI':>9s}{'Sharpe':>8s}{'maxDD':>8s}")
print('-'*69)
for k, v in carteiras.items():
    m = avaliar(v[jan], cdi[jan])
    sh = f"{m['sharpe']:>8.2f}" if np.isfinite(m['sharpe']) else f"{'-':>8s}"
    print(f"{k:28s}{m['retorno']:>9.1%}{m['vol']:>7.1%}{m['exCDI']:>+9.1%}{sh}{m['maxDD']:>8.0%}")

In [ ]:
eq = (1 + pd.DataFrame(carteiras)[jan]).cumprod()
plt.figure(); 
for c, cor in zip(eq.columns, ['#2a78d6','#1baf7a','#b4b2a9','#eda100']):
    plt.plot(eq.index, eq[c], label=c, color=cor, lw=2)
plt.yscale('log'); plt.ylabel('capital acumulado (x, escala log)')
plt.title('Robo Fundamento vs benchmarks'); plt.legend(); plt.tight_layout(); plt.show()

## 5. A alocacao ao longo do tempo (o rebalanceamento em acao)

In [ ]:
w = pesos_ranking()[jan]
plt.figure()
plt.stackplot(w.index, [w[c] for c in CLASSES], labels=CLASSES,
              colors=['#b4b2a9','#2a78d6','#1baf7a','#eda100','#e34948'])
plt.ylim(0,1); plt.ylabel('peso na carteira'); plt.title('Como o robo alocou, mes a mes')
plt.legend(loc='upper center', ncol=5, fontsize=9); plt.tight_layout(); plt.show()

print('Alocacao media:')
print((w.mean()*100).round(1).to_string())
print('Meses 100% em CDI (fuga do risco):', (w['CDI'] > 0.95).sum(), 'de', len(w))

## 6. Fora da amostra e robustez

O teste que importa: funciona em dados que nao usamos para desenhar a estrategia?

In [ ]:
oos = rets.index >= '2020-01-01'
print('FORA DA AMOSTRA (2020+):')
for k, v in carteiras.items():
    m = avaliar(v[oos], cdi[oos])
    print(f"  {k:26s} retorno={m['retorno']:>6.1%}  exCDI={m['exCDI']:>+6.1%}  Sharpe={m['sharpe']:>5.2f}  maxDD={m['maxDD']:>5.0%}")

print('\nROBUSTEZ — frequencia de rebalanceamento:')
for f, rot in [(1,'mensal'), (3,'trimestral'), (6,'semestral')]:
    m = avaliar(backtest(pesos_ranking(), freq=f)[jan], cdi[jan])
    print(f"  {rot:12s} exCDI={m['exCDI']:>+6.1%}  Sharpe={m['sharpe']:>5.2f}")

print('\nROBUSTEZ — custo de transacao:')
for c, rot in [(0.0005,'5 bps'), (0.0010,'10 bps'), (0.0025,'25 bps')]:
    m = avaliar(backtest(pesos_ranking(), custo=c)[jan], cdi[jan])
    print(f"  {rot:12s} exCDI={m['exCDI']:>+6.1%}  Sharpe={m['sharpe']:>5.2f}")

m = avaliar(carteiras['Robo Fundamento (ranking)'][jan], cdi[jan])
print(f"\nSIGNIFICANCIA: t={m['t']:.2f}, p={m['p']:.3f} -> " +
      ('significativo a 5%' if m['p']<0.05 else 'significativo a 10%' if m['p']<0.10 else 'nao significativo'))

## 7. Leitura

- O robo entrega **~+5% ao ano acima do CDI**, e bate o Ibovespa com **um terco do tombo** (-11% vs -37%).
- **Aguenta fora da amostra** (2020+), mudanca de frequencia e custos 2,5x maiores.
- A **estrutura** (rebalancear entre classes) entrega a maior parte do valor; o ranking adiciona ~1,7 p.p.
- **Honestidade:** p ~ 0,08 — significativo a 10%, nao a 5%. E o periodo favoreceu a bolsa global.

Detalhes: `docs/metodologia_robo.md` e `docs/resultados_robo.md`.